# Notebook 02 — Resource Management & Zombie Cleanup

## What you will learn
1. Why ANSYS zombie processes occur and how they block simulations
2. How to detect and kill zombie processes programmatically
3. How to estimate RAM requirements from mesh size
4. How to choose nproc (CPU cores) for your license
5. How to select the right port when multiple instances are needed

---

## Part 1: The Zombie Problem

### What is a zombie process?

In computing, a zombie process is one that has been left running after its
parent task ended or crashed.  In ANSYS:

- `launch_mapdl()` starts `ansys252.exe` in a background process
- That process opens a **gRPC server** on port 50052
- If your Python script crashes (kernel restart, Ctrl-C, power outage):
  - The Python side dies, but `ansys252.exe` keeps running
  - Port 50052 stays occupied
  - Next call to `launch_mapdl(port=50052)` fails with:
    ```
    OSError: [WinError 10048] Only one usage of each socket address is normally permitted.
    ```
  - Your ANSYS license token stays checked out

### How to think about ports

A **port** is like a numbered dock at a harbor.  When MAPDL starts, it docks
at port 50052 and waits for Python to connect.  If another ship (zombie) is
already docked there, the new ship can't land.

**Solution options:**
1. Kill the zombie (preferred — free the port for reuse)
2. Use a different port number (workaround — doesn't free the license)
3. Wait for Windows to time out the connection (takes minutes)

In [ ]:
import sys; sys.path.insert(0, '..')
from ams.resources.manager import kill_ansys_zombies, check_ports, ResourceManager

# ── 1. Check which ports are occupied ─────────────────────────────────────
print('=== Port Status ===')
port_status = check_ports()
for port, is_occupied in port_status.items():
    status = '🔴 OCCUPIED — something is listening here' if is_occupied else '🟢 free'
    print(f'  Port {port}:  {status}')

In [ ]:
# ── 2. Dry run: list what WOULD be killed ─────────────────────────────────
# Safe to run — does NOT kill anything yet
print('=== Zombie Scan (dry run) ===')
found = kill_ansys_zombies(dry_run=True, include_aedt=True, verbose=True)

print(f'\nFound {len(found)} ANSYS process(es).')
if found:
    print('\nTo kill them, run the cell below.')
else:
    print('System is clean.')

In [ ]:
# ── 3. Actually kill zombies (run only when you have no active simulations) ─
# UNCOMMENT when you want to kill processes

# killed = kill_ansys_zombies(dry_run=False, include_aedt=True, verbose=True)
# n = sum(1 for p in killed if p.get('killed'))
# print(f'Killed {n} process(es).')

print('Uncomment the kill call when ready.')

---

## Part 2: Memory Requirements

### How MAPDL uses memory

When MAPDL starts, it pre-allocates its working memory with the `-m` flag
(set via `launch_mapdl(ram=2048)`).

The global stiffness matrix **[K]** is the dominant memory consumer.  For a
structure with $n_{dof}$ degrees of freedom:

$$[K] \in \mathbb{R}^{n_{dof} \times n_{dof}}$$

But [K] is **sparse** — most entries are zero.  The number of non-zeros (nnz)
depends on the mesh connectivity:

$$\text{nnz} \approx n_{dof} \times n_{\text{bandwidth}}$$

where the bandwidth is typically $\sim 3 \times n_{\text{nodes/element}}$ for a
well-structured mesh.

Memory for the stiffness matrix (double precision, 8 bytes/entry):

$$M_{\text{stiff}} = 8 \times \text{nnz} \text{ bytes}$$

Plus overhead for the RHS vector, solution vector, preconditioner, and
post-processing arrays: typically $\sim 3 \times M_{\text{stiff}}$.

**Rule of thumb (empirical):**

$$M_{\text{MAPDL}} \approx 15 \text{ MB} \times (N_{\text{elements}} / 1000)$$

**Conservative estimate:**

$$M_{\text{MAPDL}} \approx 25 \text{ MB} \times (N_{\text{elements}} / 1000)$$

In [ ]:
from ams.resources.manager import estimate_memory
import matplotlib.pyplot as plt
import numpy as np

# ── Memory estimate for different mesh sizes ───────────────────────────────
mesh_sizes = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000]

print(f'  {"N_elements":>12}  {"Empirical (MB)":>16}  {"Conservative (MB)":>18}')
print('  ' + '-' * 50)

empirical_vals = []
conservative_vals = []

for n in mesh_sizes:
    mem = estimate_memory(n)
    empirical    = mem['total_mb']
    conservative = mem['conservative_mb']
    empirical_vals.append(empirical)
    conservative_vals.append(conservative)
    print(f'  {n:>12,}  {empirical:>16,.0f}  {conservative:>18,.0f}')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(mesh_sizes, empirical_vals,    'b-o', label='Empirical')
ax.loglog(mesh_sizes, conservative_vals, 'r--o', label='Conservative')
ax.axhline(2048, color='gray', linestyle=':', label='2 GB (typical MAPDL default)')
ax.axhline(16384, color='orange', linestyle=':', label='16 GB (large machine)')
ax.set_xlabel('Number of elements')
ax.set_ylabel('RAM estimate (MB)')
ax.set_title('MAPDL Memory Requirements vs Mesh Size')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

---

## Part 3: CPU Parallelism and Licenses

### How MAPDL parallelism works

MAPDL's parallel solver uses **MPI domain decomposition**:
1. The mesh is partitioned into $n_{\text{proc}}$ subdomains
2. Each MPI rank assembles and solves its subdomain stiffness
3. Interface DOFs are exchanged via MPI messages
4. The global solution is assembled from subdomain solutions

**Speedup** follows Amdahl's law:

$$S(n) = \frac{1}{(1 - p) + p/n}$$

where $p$ is the fraction of the work that can be parallelized ($\approx 0.85$
for most FEA solves), $n$ is the number of cores.

**Practical limit:** Beyond $n \approx 4$ cores, the MPI communication
overhead starts dominating for typical mesh sizes (< 1M elements).

### License requirements

| Configuration | License needed |
|---------------|----------------|
| nproc = 1 | ANSYS Mechanical Enterprise (1 token) |
| nproc = 2 | Mechanical Enterprise + 1 HPC token |
| nproc = 4 | Mechanical Enterprise + 3 HPC tokens |
| nproc = n | Mechanical Enterprise + (n-1) HPC tokens |

Mechanical Enterprise typically includes 4 HPC tokens.

In [ ]:
# ── Amdahl's law speedup curve ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

n_cores = np.arange(1, 17)
p_parallel = 0.85  # 85% parallelizable fraction (typical FEA)

speedup_amdahl = 1 / ((1 - p_parallel) + p_parallel / n_cores)
speedup_ideal  = n_cores

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_cores, speedup_ideal,  'k--', alpha=0.5, label='Ideal (linear)')
ax.plot(n_cores, speedup_amdahl, 'b-o', label=f'Amdahl (p={p_parallel})')
ax.axvline(4, color='red', linestyle=':', alpha=0.7, label='Typical HPC license limit')
ax.set_xlabel('Number of cores (nproc)')
ax.set_ylabel('Speedup vs single core')
ax.set_title('Parallel Speedup — Amdahl\'s Law')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('Marginal gain from adding more cores:')
for n in [1, 2, 4, 8, 16]:
    s = 1 / ((1 - p_parallel) + p_parallel / n)
    print(f'  nproc={n:2d}: {s:.2f}× speedup  ({s/n*100:.0f}% efficiency)')

In [ ]:
# ── Full resource recommendation ───────────────────────────────────────────
from ams.resources.manager import ResourceManager

rm = ResourceManager()
rm.print_report(n_elements=50_000, has_hpc_license=False)

---

## Summary

| Step | What to do | When |
|------|-----------|------|
| 1 | `check_ports()` | Before every `launch_mapdl()` |
| 2 | `kill_ansys_zombies()` | If any port is occupied |
| 3 | `estimate_memory(n_elements)` | To set `ram_mb` in config |
| 4 | `ResourceManager().recommend()` | To set nproc and port |

**Next:** `03_geometry_import_and_mesh_quality.ipynb`